# Chạy AI service (model retrain_v6.1 GGUF) trên Colab GPU

Notebook này chạy **AI Service** với **model retrain_v6.1** (đã fine-tune trên dữ liệu CV thật intern/fresher, xuất định dạng GGUF Q4_K_M) trên GPU T4 của Google Colab và mở **Cloudflare Tunnel** để backend Spring Boot ở máy local hoặc client ngoài kết nối trực tiếp. Không cần tải file model ~1.1 GB về máy local.

### Hướng dẫn chạy:
1. **Bật GPU**: Menu `Runtime` → `Change runtime type` → Chọn **T4 GPU** → Bấm `Save`.
2. **Chạy toàn bộ**: Menu `Runtime` → **Run all** (hoặc chạy lần lượt từng cell [1] đến [6]).
3. **Lấy URL Tunnel**: Sau khi Cell [4] chạy xong, copy URL dạng `https://....trycloudflare.com`.
4. **Cấu hình Spring Boot local** trỏ vào URL trên:
   ```bash
   cd backend && ./mvnw spring-boot:run -Dspring-boot.run.arguments=--ai.service.url=https://....trycloudflare.com
   ```
5. **Theo dõi**: Giữ tab Colab mở; Cell [6] sẽ hiển thị số lượng request và thời gian xử lý thực tế.


In [ ]:
# [1] Cấu hình
REPO = "https://github.com/long20102004/recruit_system.git"
BRANCH = "dev_hocnv"

# Đường dẫn lưu trữ Drive dự án (nơi retrain_qwen3_v2.ipynb xuất model v6.1)
DRIVE_ROOT = "/content/drive/MyDrive/cv-analysis-service"
OUT_DIR = f"{DRIVE_ROOT}/finetune_output_v6.1"

# Tùy chọn: ID Google Drive chia sẻ công khai nếu bạn đã tải file lên Drive và lấy link share
# Nếu đã mount Drive cá nhân có chứa thư mục finetune_output_v6.1 thì để trống ""
GGUF_DRIVE_ID = ""  # ví dụ: "1xyz..." (nếu có)


In [ ]:
# [2] Clone mã nguồn nhánh dev_hocnv + Cài đặt thư viện (llama-cpp-python CUDA 12, cloudflared)
%cd /content
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
%cd /content/recruit_system/ai-service
!git log --oneline -1

!pip install -q fastapi uvicorn pydantic json-repair rapidfuzz gdown python-dotenv
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Đăng ký thư viện CUDA của pip nvidia-* với dynamic linker ldconfig
import glob, site, os
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
!ldconfig
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])

import llama_cpp
print("llama_cpp version:", llama_cpp.__version__, "| GPU offload:", llama_cpp.llama_supports_gpu_offload())
assert llama_cpp.llama_supports_gpu_offload(), "Chưa bật GPU offload! Hãy kiểm tra lại Runtime Type (chọn T4 GPU)."


In [ ]:
# [3] Nạp model GGUF v6.1 từ Google Drive (Mount hoặc gdown)
import glob, os, subprocess

GGUF_PATH = None

# Cách 1: Nạp trực tiếp từ Google Drive đã mount (nơi retrain_v6.1 lưu)
if not GGUF_DRIVE_ID:
    if not os.path.exists("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")

    # Tìm file GGUF trong thư mục finetune_output_v6.1 hoặc DRIVE_ROOT
    candidates = sorted(
        glob.glob(f"{OUT_DIR}/gguf*/**/*.gguf", recursive=True) +
        glob.glob(f"{OUT_DIR}/gguf*.gguf") +
        glob.glob(f"{DRIVE_ROOT}/*v6.1*/**/*.gguf", recursive=True) +
        glob.glob(f"{DRIVE_ROOT}/*v6.1*.gguf")
    )
    if candidates:
        # Ưu tiên bản q4_k_m
        GGUF_PATH = next((c for c in candidates if "q4_k_m" in c.lower()), candidates[0])
        print("Đã tìm thấy GGUF v6.1 từ Google Drive:", GGUF_PATH)

# Cách 2: Nếu có GGUF_DRIVE_ID chia sẻ công khai qua gdown
if not GGUF_PATH and GGUF_DRIVE_ID:
    os.makedirs("/content/recruit_system/ai-service/merged_model", exist_ok=True)
    local_path = "/content/recruit_system/ai-service/merged_model/model-v6.1-q4_k_m.gguf"
    if not os.path.exists(local_path):
        subprocess.run(["gdown", "-q", GGUF_DRIVE_ID, "-O", local_path], check=True)
    GGUF_PATH = local_path

# Fallback: Kiểm tra trong merged_model của repo
if not GGUF_PATH:
    for fallback in [
        "/content/recruit_system/ai-service/merged_model/model-v6.1-q4_k_m.gguf",
        "/content/recruit_system/ai-service/merged_model/model-v4-q4_k_m.gguf",
    ]:
        if os.path.exists(fallback):
            GGUF_PATH = fallback
            print("Sử dụng fallback GGUF:", GGUF_PATH)
            break

assert GGUF_PATH and os.path.exists(GGUF_PATH), (
    f"Không tìm thấy file GGUF cho v6.1! Hãy đảm bảo cell xuất GGUF ở notebook train đã hoàn tất "
    f"hoặc file .gguf đã có trong Drive ({OUT_DIR})."
)

size_gb = round(os.path.getsize(GGUF_PATH) / 1e9, 2)
print(f"File GGUF đã sẵn sàng: {GGUF_PATH} ({size_gb} GB)")


In [ ]:
# [4] Khởi chạy AI Service với GGUF v6.1 + Mở Cloudflare Tunnel
%cd /content/recruit_system/ai-service
import subprocess, time, re, urllib.request, json, os

# Cấu hình môi trường: MODEL_ENGINE=llama_cpp và PROMPT_VERSION=v3 (v6.1 sử dụng prompt_v3)
env = dict(
    os.environ,
    MODEL_ENGINE="llama_cpp",
    PROMPT_VERSION="v3",
    GGUF_MODEL_PATH=GGUF_PATH,
    LD_LIBRARY_PATH=":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
)
env.pop("KB_DSN", None)  # Trên Colab dùng seed CSV, không cần Postgres

# Khởi động FastAPI service qua uvicorn
srv = subprocess.Popen(
    ["python", "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("server.log", "w"),
    stderr=subprocess.STDOUT,
    env=env
)

# Chờ service khởi động và kiểm tra health check
h = None
for _ in range(150):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3) as resp:
            h = json.load(resp)
            break
    except Exception:
        time.sleep(2)

print("Trạng thái Health:", h)
assert h and h.get("status") == "ok" and h.get("engine") == "llama_cpp", (
    "Service chưa khởi động đúng! Xem chi tiết trong server.log:\n" + open("server.log").read()[-500:]
)

# Mở Cloudflare Tunnel ra Internet
tun = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=open("tunnel.log", "w"),
    stderr=subprocess.STDOUT
)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break

assert url, "Không lấy được URL Cloudflare Tunnel. Hãy xem file tunnel.log."
print("=" * 70)
print("🚀 URL AI-SERVICE V6.1:", url)
print("👉 Lệnh chạy Backend Spring Boot local:")
print(f"   cd backend && ./mvnw spring-boot:run -Dspring-boot.run.arguments=--ai.service.url={url}")
print("=" * 70)


In [ ]:
# [5] Test thử nghiệm trích xuất một CV mẫu
import time, json, urllib.request

sample_cv = """NGUYEN VAN INTERN
intern.dev@example.com | 0987654321 | https://github.com/intern-dev
MỤC TIÊU NGHỀ NGHIỆP
Mong muốn tìm kiếm cơ hội thực tập vị trí Java Backend Developer để áp dụng kiến thức Spring Boot và học hỏi kinh nghiệm dự án thực tế.

HỌC VẤN
Học viện Công nghệ Bưu chính Viễn thông (PTIT)
Chuyên ngành: Công nghệ Thông tin | 2022 - 2026 | GPA: 3.2/4.0

KINH NGHIỆM LÀM VIỆC
Công ty Cổ phần Công nghệ ABC      06/2025 - Hiện tại
Thực tập sinh Backend (Java Intern)
• Tham gia phát triển các API quản lý người dùng sử dụng Spring Boot và PostgreSQL.
• Viết unit test với JUnit 5 và Mockito, độ bao phủ đạt 80%.

DỰ ÁN CÁ NHÂN
Hệ thống tuyển dụng thông minh (Recruit System)      01/2025 - 05/2025
Vai trò: Backend Developer | Nhóm: 4 người
• Xây dựng RESTful API với Spring Boot, Spring Security (JWT).
• Tích hợp AI microservice trích xuất CV tự động bằng FastAPI.
• Công nghệ: Java 17, Spring Boot, PostgreSQL, Docker, Redis.

KỸ NĂNG
• Ngôn ngữ: Java, SQL, Python, JavaScript
• Frameworks/Tools: Spring Boot, Hibernate, Docker, Git, Postman
• Cơ sở dữ liệu: PostgreSQL, MySQL

CHỨNG CHỈ & NGOẠI NGỮ
• Tiếng Anh: TOEIC 750 (Listening & Reading)
"""

t0 = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:8000/api/v1/extract-cv",
    data=json.dumps({"cv_text": sample_cv}).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=300) as resp:
    res = json.load(resp)

data = res.get("data", {})
elapsed = time.time() - t0

print(f"⏱️ Thời gian trích xuất: {elapsed:.2f}s")
print(f"👤 Họ tên: {data.get('personal_info', {}).get('full_name')}")
print(f"🎓 Học vấn: {[(e.get('school'), e.get('major'), e.get('gpa')) for e in data.get('education', [])]}")
print(f"💼 Kinh nghiệm: {[(e.get('company'), e.get('job_title'), e.get('start_date')) for e in data.get('experience', [])]}")
print(f"🚀 Dự án: {[(p.get('name'), p.get('role'), p.get('tech_stack')) for p in data.get('personal_projects', [])]}")
print(f"🛠️ Kỹ năng: {data.get('skills', [])}")
print(f"🌐 Ngoại ngữ: {data.get('languages', [])}")


In [ ]:
# [6] Giữ session sống + Theo dõi số lượng request (Dừng cell này khi kết thúc)
import time, re

print("Đang giữ session hoạt động... Nhấn nút Stop (■) trên cell này khi bạn muốn dừng service.")
while True:
    if srv.poll() is not None:
        print("⚠️ Cảnh báo: Service đã dừng hoạt động!")
        break
    if tun.poll() is not None:
        print("⚠️ Cảnh báo: Cloudflare tunnel đã ngắt kết nối!")
        break

    log = open("server.log", encoding="utf-8", errors="ignore").read()
    times = [float(x) for x in re.findall(r"Inference time: ([0-9.]+)s", log)]
    avg = f"{sum(times) / len(times):.2f}s" if times else "-"
    print(time.strftime("[%H:%M:%S]"), f"Requests: {len(times)} | Avg inference: {avg} | Status: Healthy", flush=True)
    time.sleep(60)
